# EWS 鲁棒副本数据集（预处理）

从 **`EWS/data/EWS-Dataset`** 复制到 **`EWS-Dataset-robust`**（`DST_DIR_NAME` 可改）。

**关于分辨率**：这里的 `IMG_SIZE` 与微调里 `dataset_ews_seg(..., img_size=IMG_SIZE)` 一致。若磁盘上的 PNG **已经是 `IMG_SIZE × IMG_SIZE`**，代码 **不会再做 resize**（避免无意义插值）；**只有宽高不一致时** 才把 RGB（双线性）和 mask（nearest）缩放到 `IMG_SIZE`，以便与训练分辨率一致，且 **`PATCH_SIZE` 网格能整除**。

- **`train` / `validation`**：**RGB** 仅必要时缩放到 `IMG_SIZE`，**不加噪、不 mask**。**Mask** 同样仅在必要时缩放，再在 **`PATCH_SIZE` 网格** 上随机 **1% patch** 内标注 **整体取反**（0↔1）。
- **`test`**：**RGB** 在 `IMG_SIZE` 上（必要时先缩放）加 **噪声** 与 **5% patch 置零**。**Mask** 仅在必要时缩放，**不篡改**。

要求：`IMG_SIZE % PATCH_SIZE == 0`。

微调时将 `dataset_root` 指向 **`.../EWS-Dataset-robust`**。

In [5]:
# ========== 配置 ==========
SRC_DIR_NAME = "EWS-Dataset"
DST_DIR_NAME = "EWS-Dataset-robust"

IMG_SIZE = 350
PATCH_SIZE = 5  # 须与微调里 patch_size 一致

# 仅用于 test RGB
NOISE_STD = 0.05
TEST_PATCH_MASK_FRACTION = 0.05  # test 图像 5% patch 置零

# 仅用于 train / val 的 mask 标注
MASK_LABEL_INVERT_FRACTION = 0.01  # 1% patch 内 0/1 取反

SEED = 42

IF_DEST_EXISTS = "abort"  # "abort" | "overwrite"

In [6]:
from __future__ import annotations

import shutil
from pathlib import Path

import numpy as np
from PIL import Image


def find_repo(start: Path) -> Path:
    for p in [start.resolve(), *start.resolve().parents]:
        if (p / "EWS" / "data" / SRC_DIR_NAME / "train").is_dir():
            return p
    raise FileNotFoundError(
        f"未找到 {SRC_DIR_NAME}/train，请在包含 EWS/data 的仓库根目录下打开本笔记本。"
    )


REPO = find_repo(Path.cwd())
SRC_ROOT = (REPO / "EWS" / "data" / SRC_DIR_NAME).resolve()
DST_ROOT = (REPO / "EWS" / "data" / DST_DIR_NAME).resolve()

if IMG_SIZE % PATCH_SIZE != 0:
    raise ValueError(f"IMG_SIZE ({IMG_SIZE}) 必须能被 PATCH_SIZE ({PATCH_SIZE}) 整除")

for split in ("train", "validation", "test"):
    if not (SRC_ROOT / split).is_dir():
        raise FileNotFoundError(f"缺少目录: {SRC_ROOT / split}")

if DST_ROOT.exists():
    if IF_DEST_EXISTS == "overwrite":
        shutil.rmtree(DST_ROOT)
    else:
        raise FileNotFoundError(
            f"目标已存在: {DST_ROOT}\n"
            f"若需覆盖，请将 IF_DEST_EXISTS 设为 \"overwrite\" 后重跑。"
        )

DST_ROOT.mkdir(parents=True, exist_ok=True)

print("SRC:", SRC_ROOT)
print("DST:", DST_ROOT)

SRC: D:\Cursor\UNSW-COMP-9517\EWS\data\EWS-Dataset
DST: D:\Cursor\UNSW-COMP-9517\EWS\data\EWS-Dataset-robust


In [7]:
def list_image_stems(split_dir: Path) -> list[str]:
    stems: list[str] = []
    for p in sorted(split_dir.glob("*.png")):
        if p.name.endswith("_mask.png"):
            continue
        mask_path = p.with_name(p.stem + "_mask.png")
        if not mask_path.is_file():
            raise FileNotFoundError(f"缺少 mask: {mask_path}")
        stems.append(p.stem)
    if not stems:
        raise FileNotFoundError(f"未找到 RGB 图: {split_dir}")
    return stems


def load_mask_01(mask_path: Path) -> np.ndarray:
    im = Image.open(mask_path)
    if im.mode not in ("L", "LA", "RGBA", "RGB"):
        im = im.convert("RGBA")
    arr = np.array(im)
    if arr.ndim == 2:
        L = arr
    else:
        L = arr[..., 0]
    return (L > 0).astype(np.float32)


def mask_01_to_img_size(mask01: np.ndarray) -> np.ndarray:
    H, W = mask01.shape[:2]
    if H == IMG_SIZE and W == IMG_SIZE:
        return mask01.astype(np.float32, copy=False)
    m = Image.fromarray((mask01 * 255).astype(np.uint8), mode="L")
    m = m.resize((IMG_SIZE, IMG_SIZE), resample=Image.Resampling.NEAREST)
    return (np.array(m) > 0).astype(np.float32)


def save_mask_png(mask01: np.ndarray, dst_mask: Path) -> None:
    Image.fromarray((mask01 * 255).astype(np.uint8), mode="L").save(dst_mask)


def invert_random_patches_in_mask(
    m: np.ndarray,
    *,
    patch_size: int,
    invert_fraction: float,
    rng: np.random.Generator,
) -> np.ndarray:
    """(H,W) float 0/1：随机 invert_fraction 比例的 patch 内全部 1-m"""
    out = m.astype(np.float32, copy=True)
    H, W = out.shape
    nh, nw = H // patch_size, W // patch_size
    n_patches = nh * nw
    k = int(round(invert_fraction * n_patches))
    k = max(0, min(n_patches, k))
    if k == 0:
        return out
    chosen = rng.choice(n_patches, size=k, replace=False)
    for pid in chosen:
        ii = int(pid // nw)
        jj = int(pid % nw)
        h0, h1 = ii * patch_size, (ii + 1) * patch_size
        w0, w1 = jj * patch_size, (jj + 1) * patch_size
        out[h0:h1, w0:w1] = 1.0 - out[h0:h1, w0:w1]
    return out


def corrupt_rgb_hwc(
    rgb01: np.ndarray,
    *,
    patch_size: int,
    mask_fraction: float,
    noise_std: float,
    rng: np.random.Generator,
) -> np.ndarray:
    x = rgb01.astype(np.float32, copy=True)
    if noise_std > 0:
        x = x + rng.normal(0.0, noise_std, x.shape).astype(np.float32)
        x = np.clip(x, 0.0, 1.0)
    if mask_fraction <= 0:
        return x
    H, W = x.shape[:2]
    nh, nw = H // patch_size, W // patch_size
    n_patches = nh * nw
    k = int(round(mask_fraction * n_patches))
    k = max(0, min(n_patches, k))
    if k == 0:
        return x
    chosen = rng.choice(n_patches, size=k, replace=False)
    patch_mask = np.zeros((nh, nw), dtype=np.float32)
    ci = chosen // nw
    cj = chosen % nw
    patch_mask[ci, cj] = 1.0
    pixel_mask = np.kron(patch_mask, np.ones((patch_size, patch_size), dtype=np.float32))
    x = x * (1.0 - pixel_mask[..., np.newaxis])
    return x


def write_clean_rgb(src_png: Path, dst_png: Path) -> None:
    im = Image.open(src_png).convert("RGB")
    if im.size != (IMG_SIZE, IMG_SIZE):
        im = im.resize((IMG_SIZE, IMG_SIZE), resample=Image.Resampling.BILINEAR)
    im.save(dst_png)


def write_train_val_pair(src_png: Path, src_mask: Path, dst_png: Path, dst_mask: Path, rng: np.random.Generator) -> None:
    write_clean_rgb(src_png, dst_png)
    m = load_mask_01(src_mask)
    m = mask_01_to_img_size(m)
    m = invert_random_patches_in_mask(
        m,
        patch_size=PATCH_SIZE,
        invert_fraction=MASK_LABEL_INVERT_FRACTION,
        rng=rng,
    )
    save_mask_png(m, dst_mask)


def write_test_pair(src_png: Path, src_mask: Path, dst_png: Path, dst_mask: Path, rng: np.random.Generator) -> None:
    im = Image.open(src_png).convert("RGB")
    if im.size != (IMG_SIZE, IMG_SIZE):
        im = im.resize((IMG_SIZE, IMG_SIZE), resample=Image.Resampling.BILINEAR)
    rgb = np.array(im).astype(np.float32) / 255.0
    rgb = corrupt_rgb_hwc(
        rgb,
        patch_size=PATCH_SIZE,
        mask_fraction=TEST_PATCH_MASK_FRACTION,
        noise_std=NOISE_STD,
        rng=rng,
    )
    Image.fromarray(np.clip(rgb * 255.0, 0, 255).astype(np.uint8), mode="RGB").save(dst_png)
    m = load_mask_01(src_mask)
    m = mask_01_to_img_size(m)
    save_mask_png(m, dst_mask)

In [8]:
rng = np.random.default_rng(SEED)
pair_counts: dict[str, int] = {}

for split in ("train", "validation", "test"):
    src_sp = SRC_ROOT / split
    dst_sp = DST_ROOT / split
    dst_sp.mkdir(parents=True, exist_ok=True)
    stems = list_image_stems(src_sp)

    for stem in stems:
        sp = src_sp / f"{stem}.png"
        sm = src_sp / f"{stem}_mask.png"
        dp = dst_sp / f"{stem}.png"
        dm = dst_sp / f"{stem}_mask.png"
        if split == "test":
            write_test_pair(sp, sm, dp, dm, rng)
        else:
            write_train_val_pair(sp, sm, dp, dm, rng)
    pair_counts[split] = len(stems)

print("完成。各 split 样本对数:", pair_counts)
print("train/val: 干净 RGB + mask 随机 1% patch 取反；test: 噪声+5% patch mask RGB + 干净 mask")
print("dataset_root:", DST_ROOT)

完成。各 split 样本对数: {'train': 142, 'validation': 24, 'test': 24}
train/val: 干净 RGB + mask 随机 1% patch 取反；test: 噪声+5% patch mask RGB + 干净 mask
dataset_root: D:\Cursor\UNSW-COMP-9517\EWS\data\EWS-Dataset-robust
